# LeetCode 3777 — Minimum Deletions to Make Alternating Substring bằng Python

- Link: https://leetcode.com/problems/minimum-deletions-to-make-alternating-substring/
- Độ khó: **Hard**
- Chủ đề: Chuỗi, biến đổi bài toán thành **đếm cặp kề bằng nhau**, Fenwick Tree (BIT), Segment Tree, chia căn (sqrt decomposition)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minDeletions(self, s: str, queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Chứng minh: số lần xoá tối thiểu để `s[l..r]` xen kẽ = **số cặp kề bằng nhau** trong đoạn, tức số `i ∈ [l+1, r]` với `s[i] = s[i−1]`.
2. Biến bài toán thành **mảng cờ** `bad[i] = [s[i] == s[i−1]]` với truy vấn **tổng đoạn** và cập nhật **điểm**.
3. Nhận ra lật `s[j]` chỉ đảo **hai cờ** `bad[j]` và `bad[j+1]`.
4. Cài Fenwick Tree, Segment Tree và chia căn; so sánh với brute force `O(n)` mỗi truy vấn.
5. Kiểm chứng bằng định nghĩa gốc (DP xoá tối thiểu) trên dữ liệu ngẫu nhiên.

## 2. Đề bài & ví dụ

Chuỗi `s` gồm `'A'`, `'B'`. Truy vấn:

- `[1, j]`: lật `s[j]` (`A ↔ B`), **thay đổi** chuỗi cho các truy vấn sau.
- `[2, l, r]`: số ký tự **ít nhất** cần xoá để `s[l..r]` thành xen kẽ (không có hai ký tự kề bằng nhau). Không thay đổi `s`.

Trả về mảng đáp án các truy vấn loại 2.

**Ví dụ 1:** `s = "ABA"`, `[[2,1,2],[1,1],[2,0,2]]` → `[0, 2]`.

**Ví dụ 2:** `s = "ABB"`, `[[2,0,2],[1,2],[2,0,2]]` → `[1, 0]`.

**Ví dụ 3:** `s = "BABA"`, `[[2,0,3],[1,1],[2,1,3]]` → `[0, 1]`.

**Ràng buộc:** `1 ≤ n, q ≤ 10^5`.

In [1]:
from typing import List
from functools import lru_cache
import random, time, math

examples = [
    ("ABA", [[2, 1, 2], [1, 1], [2, 0, 2]], [0, 2]),
    ("ABB", [[2, 0, 2], [1, 2], [2, 0, 2]], [1, 0]),
    ("BABA", [[2, 0, 3], [1, 1], [2, 1, 3]], [0, 1]),
]
for s, qs, exp in examples:
    print(s, qs, "-> kỳ vọng", exp)

ABA [[2, 1, 2], [1, 1], [2, 0, 2]] -> kỳ vọng [0, 2]
ABB [[2, 0, 2], [1, 2], [2, 0, 2]] -> kỳ vọng [1, 0]
BABA [[2, 0, 3], [1, 1], [2, 1, 3]] -> kỳ vọng [0, 1]


## 3. Kiến thức nền

### 3.1. Vì sao "số cặp kề bằng nhau" là đáp án

Chia đoạn thành các **khối** ký tự giống nhau liên tiếp, ví dụ `AAABBA` = `AAA|BB|A`. Chuỗi xen kẽ con dài nhất giữ đúng **một ký tự mỗi khối** (hai khối kề luôn khác ký tự). Vậy số phải xoá = độ dài − số khối = Σ(độ dài khối − 1) = **số cặp kề bằng nhau**.

In [2]:
from itertools import groupby
t = "AAABBA"
blocks = ["".join(g) for _, g in groupby(t)]
pairs = sum(t[i] == t[i - 1] for i in range(1, len(t)))
print("khối:", blocks, "| len - số khối =", len(t) - len(blocks), "| số cặp kề bằng nhau =", pairs)

khối: ['AAA', 'BB', 'A'] | len - số khối = 3 | số cặp kề bằng nhau = 3


### 3.2. Fenwick Tree (Binary Indexed Tree)

Hỗ trợ `update(i, delta)` và `prefix(i)` trong `O(log n)`; tổng đoạn `[a, b]` = `prefix(b) − prefix(a−1)`. Chỉ số trong BIT bắt đầu từ 1.

In [3]:
class BIT:
    def __init__(self, n):
        self.n = n; self.c = [0] * (n + 1)
    def update(self, i, d):                 # i: 1-indexed
        while i <= self.n:
            self.c[i] += d; i += i & -i
    def prefix(self, i):
        s = 0
        while i > 0:
            s += self.c[i]; i -= i & -i
        return s

b = BIT(8)
for i, v in enumerate([0, 1, 0, 1, 1, 0, 0, 1], 1):
    if v: b.update(i, v)
print("tổng [2..5] =", b.prefix(5) - b.prefix(1))

tổng [2..5] = 3


### 3.3. Lật một ký tự ảnh hưởng tới đâu?

Cờ `bad[i]` chỉ phụ thuộc `s[i−1]` và `s[i]`. Lật `s[j]` thay đổi đúng `bad[j]` (nếu `j ≥ 1`) và `bad[j+1]` (nếu `j+1 < n`) — và mỗi cờ **đảo giá trị** (vì một trong hai ký tự so sánh đổi, ký tự kia giữ nguyên, trong bảng chữ cái chỉ hai ký tự).

In [4]:
s = list("ABBA")
flags = lambda s: [0] + [int(s[i] == s[i - 1]) for i in range(1, len(s))]
print("trước:", "".join(s), flags(s))
s[2] = 'A' if s[2] == 'B' else 'B'
print("lật j=2:", "".join(s), flags(s), "← chỉ bad[2], bad[3] đổi")

trước: ABBA [0, 0, 1, 0]
lật j=2: ABAA [0, 0, 0, 1] ← chỉ bad[2], bad[3] đổi


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

- Đáp án truy vấn `[2, l, r]` = `Σ bad[i]` với `i ∈ [l+1, r]` (cặp `(i−1, i)` phải nằm trọn trong đoạn — nên bắt đầu từ `l+1`).
- Truy vấn `[1, j]` = đảo `bad[j]` và `bad[j+1]`.

Đây là bài **cập nhật điểm + tổng đoạn** kinh điển ⇒ Fenwick / Segment Tree, `O((n + q) log n)`.

Ví dụ 1 sau khi lật: `"AAA"`, `bad = [0,1,1]`, truy vấn `[0,2]` ⇒ `bad[1] + bad[2] = 2`. ✔

## 5. Cách 1 — Brute Force: đếm trực tiếp mỗi truy vấn

In [5]:
def brute(s, queries):
    s = list(s); ans = []
    for q in queries:
        if q[0] == 1:
            j = q[1]; s[j] = 'A' if s[j] == 'B' else 'B'
        else:
            _, l, r = q
            ans.append(sum(s[i] == s[i - 1] for i in range(l + 1, r + 1)))
    return ans

for s, qs, exp in examples:
    print(brute(s, qs), "kỳ vọng", exp)

[0, 2] kỳ vọng [0, 2]
[1, 0] kỳ vọng [1, 0]
[0, 1] kỳ vọng [0, 1]


**Nút thắt:** `O(n)` mỗi truy vấn ⇒ `O(n·q) = 10^10` trường hợp xấu nhất.

Để chắc chắn công thức "đếm cặp" đúng với **định nghĩa gốc**, ta còn có một kiểm chứng độc lập: DP tìm dãy con xen kẽ dài nhất.

In [6]:
def min_del_dp(t):
    # dp[c] = độ dài dãy con xen kẽ dài nhất kết thúc bằng ký tự c
    dp = {'A': 0, 'B': 0}
    for ch in t:
        other = 'B' if ch == 'A' else 'A'
        dp[ch] = max(dp[ch], dp[other] + 1)
    return len(t) - max(dp.values())

def brute_dp(s, queries):
    s = list(s); ans = []
    for q in queries:
        if q[0] == 1:
            j = q[1]; s[j] = 'A' if s[j] == 'B' else 'B'
        else:
            ans.append(min_del_dp(s[q[1]:q[2] + 1]))
    return ans

print([brute_dp(s, qs) for s, qs, _ in examples])

[[0, 2], [1, 0], [0, 1]]


## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát biến bài **chuỗi** thành bài **mảng số**:

1. Đáp án = tổng `bad` trên `[l+1, r]` (mục 3.1).
2. Lật chỉ đảo hai cờ (mục 3.3).

Sau đó mọi cấu trúc "point update / range sum" đều dùng được. Với `bad[i] ∈ {0,1}`, cập nhật là `delta = 1 − 2·bad[i]`.

## 7. Các cách tiếp theo

- **Cách 2:** chia căn — khối kích thước `B ≈ √n`, lưu tổng từng khối: cập nhật `O(1)`, truy vấn `O(√n)`.
- **Cách 3:** Fenwick Tree — `O(log n)` cả hai thao tác (lời giải nộp).
- **Cách 4:** Segment Tree lặp (bottom-up) — `O(log n)`.

### 7.1. Cách 2 — Chia căn

In [7]:
def sqrt_decomp(s, queries):
    n = len(s); s = list(s)
    bad = [0] + [int(s[i] == s[i - 1]) for i in range(1, n)]
    B = max(1, int(math.isqrt(n)))
    blk = [0] * (n // B + 1)
    for i, v in enumerate(bad):
        blk[i // B] += v

    def toggle(i):
        d = 1 - 2 * bad[i]; bad[i] ^= 1; blk[i // B] += d

    def range_sum(a, b):                     # tổng bad[a..b], a <= b
        res = 0
        while a <= b and a % B:
            res += bad[a]; a += 1
        while a + B - 1 <= b:
            res += blk[a // B]; a += B
        while a <= b:
            res += bad[a]; a += 1
        return res

    ans = []
    for q in queries:
        if q[0] == 1:
            j = q[1]
            if j >= 1: toggle(j)
            if j + 1 < n: toggle(j + 1)
        else:
            _, l, r = q
            ans.append(range_sum(l + 1, r) if l < r else 0)
    return ans

print([sqrt_decomp(s, qs) for s, qs, _ in examples])

[[0, 2], [1, 0], [0, 1]]


**Lưu ý:** cờ `bad[0]` luôn bằng 0 và không bao giờ được cộng vào đáp án (đoạn bắt đầu từ `l+1 ≥ 1`), nên có thể bỏ qua việc cập nhật nó khi `j = 0`.

### 7.2. Cách 3 — Fenwick Tree (lời giải chính)

In [8]:
def fenwick_solve(s, queries):
    n = len(s)
    bad = [0] * n
    bit = BIT(n)
    for i in range(1, n):
        if s[i] == s[i - 1]:
            bad[i] = 1; bit.update(i + 1, 1)          # bad[i] ở vị trí i+1 của BIT
    ans = []
    for q in queries:
        if q[0] == 1:
            j = q[1]
            for i in (j, j + 1):
                if 1 <= i < n:
                    bit.update(i + 1, 1 - 2 * bad[i]); bad[i] ^= 1
        else:
            _, l, r = q
            ans.append(bit.prefix(r + 1) - bit.prefix(l + 1))   # bad[l+1..r]
    return ans

print([fenwick_solve(s, qs) for s, qs, _ in examples])

[[0, 2], [1, 0], [0, 1]]


### 7.3. Cách 4 — Segment Tree bottom-up

In [9]:
def segtree_solve(s, queries):
    n = len(s)
    size = 1
    while size < n: size *= 2
    t = [0] * (2 * size)
    for i in range(1, n):
        t[size + i] = int(s[i] == s[i - 1])
    for i in range(size - 1, 0, -1):
        t[i] = t[2 * i] + t[2 * i + 1]

    def toggle(i):
        p = size + i; t[p] ^= 1; p //= 2
        while p:
            t[p] = t[2 * p] + t[2 * p + 1]; p //= 2

    def query(a, b):                        # [a, b] bao gồm hai đầu
        res = 0; a += size; b += size + 1
        while a < b:
            if a & 1: res += t[a]; a += 1
            if b & 1: b -= 1; res += t[b]
            a //= 2; b //= 2
        return res

    ans = []
    for q in queries:
        if q[0] == 1:
            j = q[1]
            if j >= 1: toggle(j)
            if j + 1 < n: toggle(j + 1)
        else:
            _, l, r = q
            ans.append(query(l + 1, r) if l < r else 0)
    return ans

print([segtree_solve(s, qs) for s, qs, _ in examples])

[[0, 2], [1, 0], [0, 1]]


## 8. TRACE từng bước

In [10]:
def trace_fenwick(s, queries):
    n = len(s); s = list(s)
    bad = [0] + [int(s[i] == s[i - 1]) for i in range(1, n)]
    bit = BIT(n)
    for i in range(1, n):
        if bad[i]: bit.update(i + 1, 1)
    print("s =", "".join(s), "| bad =", bad)
    for q in queries:
        if q[0] == 1:
            j = q[1]; s[j] = 'A' if s[j] == 'B' else 'B'
            changed = []
            for i in (j, j + 1):
                if 1 <= i < n:
                    bit.update(i + 1, 1 - 2 * bad[i]); bad[i] ^= 1; changed.append(i)
            print(f"lật j={j}: s = {''.join(s)} | đảo bad{changed} -> bad = {bad}")
        else:
            _, l, r = q
            res = bit.prefix(r + 1) - bit.prefix(l + 1)
            print(f"hỏi [{l},{r}] = '{''.join(s[l:r+1])}': tổng bad[{l+1}..{r}] = {res}")

trace_fenwick("ABA", [[2, 1, 2], [1, 1], [2, 0, 2]])

s = ABA | bad = [0, 0, 0]
hỏi [1,2] = 'BA': tổng bad[2..2] = 0
lật j=1: s = AAA | đảo bad[1, 2] -> bad = [0, 1, 1]
hỏi [0,2] = 'AAA': tổng bad[1..2] = 2


In [11]:
trace_fenwick("AABBBA", [[2, 0, 5], [1, 3], [2, 1, 4], [1, 0], [2, 0, 1], [2, 3, 3]])

s = AABBBA | bad = [0, 1, 0, 1, 1, 0]
hỏi [0,5] = 'AABBBA': tổng bad[1..5] = 3
lật j=3: s = AABABA | đảo bad[3, 4] -> bad = [0, 1, 0, 0, 0, 0]
hỏi [1,4] = 'ABAB': tổng bad[2..4] = 0
lật j=0: s = BABABA | đảo bad[1] -> bad = [0, 0, 0, 0, 0, 0]
hỏi [0,1] = 'BA': tổng bad[1..1] = 0
hỏi [3,3] = 'A': tổng bad[4..3] = 0


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def minDeletions(self, s: str, queries: List[List[int]]) -> List[int]:
        n = len(s)
        c = [0] * (n + 1)

        def update(i, d):
            while i <= n:
                c[i] += d; i += i & -i

        def prefix(i):
            r = 0
            while i > 0:
                r += c[i]; i -= i & -i
            return r

        bad = [0] * n
        for i in range(1, n):
            if s[i] == s[i - 1]:
                bad[i] = 1; update(i + 1, 1)
        ans = []
        for q in queries:
            if q[0] == 1:
                j = q[1]
                if j >= 1:
                    update(j + 1, 1 - 2 * bad[j]); bad[j] ^= 1
                if j + 1 < n:
                    update(j + 2, 1 - 2 * bad[j + 1]); bad[j + 1] ^= 1
            else:
                l, r = q[1], q[2]
                ans.append(prefix(r + 1) - prefix(l + 1))
        return ans

sol = Solution()
for s, qs, exp in examples:
    got = sol.minDeletions(s, qs)
    print(s, "->", got, "✔" if got == exp else "✘")

ABA -> [0, 2] ✔
ABB -> [1, 0] ✔
BABA -> [0, 1] ✔


## 10. Vì sao thuật toán đúng

1. **Công thức đáp án.** Với chuỗi `t` có `b` khối, dãy con xen kẽ dài nhất có độ dài đúng `b`: (≤) hai ký tự cùng khối không thể cùng xuất hiện kề nhau trong dãy con mà không có ký tự khác xen giữa — nhưng giữa hai phần tử cùng khối không có ký tự khác ⇒ mỗi khối đóng góp tối đa 1; (≥) lấy một ký tự mỗi khối, hai khối kề khác ký tự ⇒ xen kẽ. Do đó số xoá = `|t| − b` = số cặp kề bằng nhau.
2. **Bất biến cấu trúc.** Sau mỗi truy vấn, `bad[i] = [s[i] == s[i−1]]` với mọi `i ≥ 1` và BIT lưu đúng tổng tiền tố của `bad` — vì lật `s[j]` chỉ làm đổi `bad[j]`, `bad[j+1]`, và ta cập nhật BIT với đúng `delta`.
3. **Truy vấn.** Các cặp nằm trọn trong `[l, r]` là `(i−1, i)` với `l+1 ≤ i ≤ r` ⇒ tổng `bad[l+1..r]` = `prefix(r+1) − prefix(l+1)` (chuyển sang chỉ số BIT 1-indexed). Khi `l = r`, hiệu bằng 0.

## 11. Phân tích Time / Space Complexity

| Cách | Khởi tạo | Cập nhật | Truy vấn | Space |
|---|---|---|---|---|
| 1. Brute | `O(1)` | `O(1)` | `O(n)` | `O(n)` |
| 2. Chia căn | `O(n)` | `O(1)` | `O(√n)` | `O(n)` |
| 3. Fenwick | `O(n log n)` | `O(log n)` | `O(log n)` | `O(n)` |
| 4. Segment Tree | `O(n)` | `O(log n)` | `O(log n)` | `O(n)` |

Tổng với Fenwick: `O((n + q) log n)`.

## 12. Bộ test

In [13]:
METHODS = [brute, brute_dp, sqrt_decomp, fenwick_solve, segtree_solve, lambda s, q: Solution().minDeletions(s, q)]

for s, qs, exp in examples:
    assert all(m(s, qs) == exp for m in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên
assert Solution().minDeletions("A", [[2, 0, 0], [1, 0], [2, 0, 0]]) == [0, 0]       # n = 1
assert Solution().minDeletions("AAAA", [[2, 0, 3], [2, 1, 1], [1, 3], [2, 0, 3]]) == [3, 0, 2]
assert Solution().minDeletions("ABAB", [[1, 0], [2, 0, 3], [1, 0], [2, 0, 3]]) == [1, 0]
print("Ca biên: OK")

Ca biên: OK


In [15]:
# 12.3 ngẫu nhiên: mọi cách phải khớp, kể cả với DP định nghĩa gốc
def rand_case(rng, n, q):
    s = "".join(rng.choice("AB") for _ in range(n)); qs = []
    for _ in range(q):
        if rng.random() < 0.5:
            qs.append([1, rng.randrange(n)])
        else:
            l = rng.randrange(n); r = rng.randrange(l, n); qs.append([2, l, r])
    return s, qs

rng = random.Random(3777)
for _ in range(1500):
    s, qs = rand_case(rng, rng.randint(1, 12), rng.randint(1, 15))
    res = [m(s, qs) for m in METHODS]
    assert all(r == res[0] for r in res), (s, qs, res)
print("1500 test ngẫu nhiên: OK")

1500 test ngẫu nhiên: OK


In [16]:
# 12.4 test lớn: n = q = 10^5
rng = random.Random(99)
s, qs = rand_case(rng, 10**5, 10**5)
t0 = time.time(); a1 = Solution().minDeletions(s, qs); t1 = time.time()
a2 = segtree_solve(s, qs)
print("khớp segment tree:", a1 == a2, "| thời gian Fenwick:", round(t1 - t0, 2), "s")

khớp segment tree: True | thời gian Fenwick: 0.25 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng cả `bad[l]` (cặp `(l−1, l)` nằm ngoài đoạn)

In [17]:
def wrong_include_l(s, queries):
    s = list(s); ans = []
    for q in queries:
        if q[0] == 1:
            j = q[1]; s[j] = 'A' if s[j] == 'B' else 'B'
        else:
            _, l, r = q
            ans.append(sum(s[i] == s[i - 1] for i in range(max(l, 1), r + 1)))   # SAI: từ l
    return ans

print("sai:", wrong_include_l("ABB", [[2, 2, 2]]), "| đúng:", Solution().minDeletions("ABB", [[2, 2, 2]]))

sai: [1] | đúng: [0]


### Lỗi 2 — Chỉ cập nhật một cờ khi lật

Quên `bad[j+1]` (hoặc `bad[j]`) ⇒ BIT sai lệch dần.

In [18]:
def wrong_one_flag(s, queries):
    n = len(s); bad = [0] + [int(s[i] == s[i - 1]) for i in range(1, n)]; ans = []
    for q in queries:
        if q[0] == 1:
            j = q[1]
            if j >= 1: bad[j] ^= 1                               # SAI: quên bad[j+1]
        else:
            ans.append(sum(bad[q[1] + 1:q[2] + 1]))
    return ans

s, qs = "ABA", [[1, 1], [2, 0, 2]]
print("sai:", wrong_one_flag(s, qs), "| đúng:", Solution().minDeletions(s, qs))

sai: [1] | đúng: [2]


### Lỗi 3 — Tính lại cờ từ chuỗi gốc thay vì chuỗi đã lật

Nếu chỉ lưu `bad` mà không cập nhật `s` rồi lại tính `bad[j] = s[j] == s[j−1]` từ `s` **cũ** ⇒ sai. Cách an toàn: hoặc đảo cờ (`^= 1`), hoặc cập nhật `s` trước rồi tính lại.

In [19]:
def wrong_stale(s, queries):
    n = len(s); bad = [0] + [int(s[i] == s[i - 1]) for i in range(1, n)]; ans = []
    for q in queries:
        if q[0] == 1:
            j = q[1]                                            # SAI: không cập nhật s
            for i in (j, j + 1):
                if 1 <= i < n: bad[i] = int(s[i] == s[i - 1])
        else:
            ans.append(sum(bad[q[1] + 1:q[2] + 1]))
    return ans

s, qs = "ABB", [[1, 2], [2, 0, 2]]
print("sai:", wrong_stale(s, qs), "| đúng:", Solution().minDeletions(s, qs))

sai: [1] | đúng: [0]


### Lỗi 4 — Nghĩ phải xoá để chuỗi thành `ABAB…` hoặc `BABA…` cố định

Đề chỉ yêu cầu **không có hai ký tự kề bằng nhau** sau khi xoá; ký tự đầu tuỳ ý. Đếm số vị trí lệch so với mẫu cố định là bài khác (đổi ký tự, không phải xoá).

In [20]:
def wrong_pattern(t):
    m1 = sum(ch != "AB"[i % 2] for i, ch in enumerate(t))
    m2 = sum(ch != "BA"[i % 2] for i, ch in enumerate(t))
    return min(m1, m2)                                          # SAI: đây là số lần ĐỔI ký tự

for t in ["AAA", "ABBA"]:
    print(repr(t), ": sai", wrong_pattern(t), "| đúng", min_del_dp(t))
assert wrong_pattern("AAA") != min_del_dp("AAA") and wrong_pattern("ABBA") != min_del_dp("ABBA")

'AAA' : sai 1 | đúng 2
'ABBA' : sai 2 | đúng 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xoá ít nhất để không có hai phần tử kề bằng nhau" ⇒ **số cặp kề bằng nhau** (= độ dài − số khối).
- Truy vấn đoạn trên **cặp kề** ⇒ đặt cờ ở phần tử **sau** của cặp, đoạn trở thành `[l+1, r]`.
- Cập nhật điểm làm thay đổi `O(1)` cờ lân cận ⇒ BIT / Segment Tree.
- Luôn kiểm chứng công thức rút gọn bằng định nghĩa gốc (DP) trên dữ liệu nhỏ.

## 15. Biến thể của bài

1. **Bảng chữ cái lớn hơn** (`'a'..'z'`): công thức vẫn là số cặp kề bằng nhau; nhưng lật thành ký tự **bất kỳ** ⇒ phải tính lại cờ thay vì đảo.
2. **Lật cả đoạn** `[1, l, r]`: các cờ bên trong đoạn không đổi, chỉ `bad[l]` và `bad[r+1]` đổi!
3. **Đổi (thay) ký tự ít nhất** để xen kẽ: so với hai mẫu cố định, dùng hai BIT.
4. **Không có cập nhật:** chỉ cần mảng tổng tiền tố.

In [21]:
# Biến thể 2: lật cả đoạn — chỉ hai cờ ở biên đổi
def range_flip_solve(s, queries):
    n = len(s); bad = [0] + [int(s[i] == s[i - 1]) for i in range(1, n)]
    bit = BIT(n)
    for i in range(1, n):
        if bad[i]: bit.update(i + 1, 1)
    ans = []
    for q in queries:
        if q[0] == 1:
            _, l, r = q
            for i in (l, r + 1):
                if 1 <= i < n: bit.update(i + 1, 1 - 2 * bad[i]); bad[i] ^= 1
        else:
            ans.append(bit.prefix(q[2] + 1) - bit.prefix(q[1] + 1))
    return ans

def range_flip_brute(s, queries):
    s = list(s); ans = []
    for q in queries:
        if q[0] == 1:
            for j in range(q[1], q[2] + 1): s[j] = 'A' if s[j] == 'B' else 'B'
        else:
            ans.append(sum(s[i] == s[i - 1] for i in range(q[1] + 1, q[2] + 1)))
    return ans

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 10); s = "".join(rng.choice("AB") for _ in range(n)); qs = []
    for _ in range(10):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([rng.choice([1, 2]), l, r])
    assert range_flip_solve(s, qs) == range_flip_brute(s, qs)
print("Biến thể 2 (lật đoạn): OK")

Biến thể 2 (lật đoạn): OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Truy vấn | Cập nhật | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Đếm trực tiếp | `O(n)` | `O(1)` | Kiểm chứng |
| 2. Chia căn | Tổng theo khối | `O(√n)` | `O(1)` | Khi cần cấu trúc dễ cài |
| 3. Fenwick | Tổng tiền tố động | `O(log n)` | `O(log n)` | **Lời giải chuẩn** |
| 4. Segment Tree | Cây tổng | `O(log n)` | `O(log n)` | Khi cần mở rộng (lật đoạn, …) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `minDeletions("AABBA", [[2,0,4]])` = ?

**Bài 2 (tính tay).** `s = "ABAB"`, truy vấn `[[1,1],[2,0,3],[1,2],[2,0,3]]`.

**Bài 3 (điền TODO).** Viết `min_del_static(s, queries)` khi không có truy vấn loại 1, dùng mảng tổng tiền tố.

**Bài 4 (tự viết lại).** Viết lại lời giải bằng Segment Tree đệ quy.

**Bài 5 (mini challenge).** Bảng chữ cái `'a'..'z'`, truy vấn `[1, j, c]` gán `s[j] = c`. Viết lời giải BIT.

In [22]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_del_static(s, queries):
    # TODO: pre[i] = số cặp bằng nhau trong s[0..i]
    pass

In [23]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def min_del_alpha(s, queries):
    # TODO: cập nhật bằng cách TÍNH LẠI bad[j], bad[j+1] chứ không đảo
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cặp bằng nhau: `(0,1)`, `(2,3)`. (2) Đếm. (3) `2`.

**Bài 2.** (1) Sau lật 1: `"AAAB"` ⇒ 2. (2) Lật 2: `"AABB"`. (3) ⇒ 2. Đáp án `[2, 2]`.

**Bài 3.** (1) `pre[0] = 0`. (2) `pre[i] = pre[i−1] + [s[i]==s[i−1]]`. (3) Đáp án `pre[r] − pre[l]`.

**Bài 4.** (1) `build(node, lo, hi)`. (2) `update` đệ quy xuống lá. (3) `query` gộp hai con.

**Bài 5.** (1) Lưu `s` dạng list. (2) Gán rồi tính lại hai cờ, `delta = mới − cũ`. (3) Đối chiếu brute.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().minDeletions("AABBA", [[2, 0, 4]]) == [2] == brute("AABBA", [[2, 0, 4]])
assert Solution().minDeletions("ABAB", [[1, 1], [2, 0, 3], [1, 2], [2, 0, 3]]) == [2, 2]
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
# Bài 3, 4
def min_del_static_sol(s, queries):
    pre = [0] * len(s)
    for i in range(1, len(s)):
        pre[i] = pre[i - 1] + (s[i] == s[i - 1])
    return [pre[r] - pre[l] for _, l, r in queries]

def segtree_rec(s, queries):
    n = len(s); s = list(s)
    bad = [0] + [int(s[i] == s[i - 1]) for i in range(1, n)]
    tree = [0] * (4 * n)
    def build(x, lo, hi):
        if lo == hi: tree[x] = bad[lo]; return
        m = (lo + hi) // 2; build(2 * x, lo, m); build(2 * x + 1, m + 1, hi); tree[x] = tree[2 * x] + tree[2 * x + 1]
    def upd(x, lo, hi, i):
        if lo == hi: tree[x] = bad[lo]; return
        m = (lo + hi) // 2
        upd(2 * x, lo, m, i) if i <= m else upd(2 * x + 1, m + 1, hi, i)
        tree[x] = tree[2 * x] + tree[2 * x + 1]
    def qry(x, lo, hi, a, b):
        if b < lo or hi < a: return 0
        if a <= lo and hi <= b: return tree[x]
        m = (lo + hi) // 2
        return qry(2 * x, lo, m, a, b) + qry(2 * x + 1, m + 1, hi, a, b)
    build(1, 0, n - 1); ans = []
    for q in queries:
        if q[0] == 1:
            for i in (q[1], q[1] + 1):
                if 1 <= i < n: bad[i] ^= 1; upd(1, 0, n - 1, i)
        else:
            ans.append(qry(1, 0, n - 1, q[1] + 1, q[2]) if q[1] < q[2] else 0)
    return ans

rng = random.Random(4)
for _ in range(500):
    s, qs = rand_case(rng, rng.randint(1, 10), 10)
    assert segtree_rec(s, qs) == Solution().minDeletions(s, qs)
    st = [q for q in qs if q[0] == 2]
    assert min_del_static_sol(s, st) == brute(s, st)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [26]:
# Bài 5: bảng chữ cái lớn, gán ký tự
def min_del_alpha_sol(s, queries):
    n = len(s); s = list(s)
    bad = [0] + [int(s[i] == s[i - 1]) for i in range(1, n)]
    bit = BIT(n)
    for i in range(1, n):
        if bad[i]: bit.update(i + 1, 1)
    ans = []
    for q in queries:
        if q[0] == 1:
            _, j, ch = q; s[j] = ch
            for i in (j, j + 1):
                if 1 <= i < n:
                    nv = int(s[i] == s[i - 1])
                    if nv != bad[i]: bit.update(i + 1, nv - bad[i]); bad[i] = nv
        else:
            ans.append(bit.prefix(q[2] + 1) - bit.prefix(q[1] + 1))
    return ans

def alpha_brute(s, queries):
    s = list(s); ans = []
    for q in queries:
        if q[0] == 1: s[q[1]] = q[2]
        else: ans.append(sum(s[i] == s[i - 1] for i in range(q[1] + 1, q[2] + 1)))
    return ans

rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 10); s = "".join(rng.choice("abc") for _ in range(n)); qs = []
    for _ in range(10):
        if rng.random() < 0.5: qs.append([1, rng.randrange(n), rng.choice("abc")])
        else:
            l = rng.randrange(n); qs.append([2, l, rng.randrange(l, n)])
    assert min_del_alpha_sol(s, qs) == alpha_brute(s, qs)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
bad[i] = s[i] == s[i-1]            # cờ đặt ở phần tử SAU của cặp
# lật j: đảo bad[j] (j>=1) và bad[j+1] (j+1<n), cập nhật BIT với delta = 1 - 2*bad
# hỏi [l, r]: prefix(r+1) - prefix(l+1)   (tổng bad[l+1..r], BIT 1-indexed)
```

**Checklist:**

- [ ] Đáp án = số cặp kề bằng nhau trong đoạn?
- [ ] Đoạn cờ là `[l+1, r]`, không phải `[l, r]`?
- [ ] Lật cập nhật **cả hai** cờ, có kiểm tra biên `j ≥ 1`, `j+1 < n`?
- [ ] Chỉ số BIT lệch 1 đúng chưa?
- [ ] Đã so với DP định nghĩa gốc?